Si sviluppano due idee centrali:
1. La sicurezza dipende dal modello di attaccante
2. Un protocollo sicuro per autenticare una sola parte non diventa autmaticamente sicuro se lo eseguiamo in entrambe le direzioni.

Si userà $p$ per indicare la password, $K$ per il segreto condiviso e $C$ per la challenge. Le scritture $H(K,C)$ indicheranno in generale una risposta crittografica dipendente da $K$ e $C$, non sono da interpretare necessariamente come un invito a costruire un MAC concatenando liberamente chiave e messaggio.

Distinguiamo ora tre terminologie simili, ma diverse:
- l'**Identificazione** consiste nel dichiarare quale identità digitale si vuole utilizzare (es. sono l'utente con username Mario)
- l'**Autenticazione** consiste nel dimostrare di possedere l'identità dichiarata (es. dimostro di essere Mario fornendo la password corretta)
- l'**Autorizzazione** riguarda le operazioni che quell'utente può compiere dopo l'autenticazione

Autenticare un account è diverso dall'autenticare messaggi: finora abbiamo verificato l'origine e l'integrità di un contenuto, ora vogliamo verificare **chi (in termini di identità digitale) sta partecipando a una sessione**.

Per autenticarsi moltissimi mezzi possibili, ad esempio **qualcosa che so** (password, pin, chiave segreta..), **qualcosa che ho** (token, smart card, telefono..), **qualcosa che sono** (impronta digitale, riconoscimento facciale..), **qualcosa che faccio** (pattern di digitazione, movimenti del mouse) etc...

Chiaramente esperimenti legati ai modi di muovere il mouse o usare il telefono aiutano a riconoscere l'utente hanno margini di errore: il comportamento può cambiare o il dispositivo può essere temporaneamente usato da qualcun altro.

Un altro modo è usare le **PUF** (Physical Unclonable Functions), che sfruttano le caratteristiche fisiche uniche di un dispositivo legate anche a variazioni di fabbricazione, per ottenere risposte difficili da riprodurre su un altro dispositivo.

Combinare fattori di categorie diverse porta all'**autenticazione multifattore**. Due password invece rimangono elementi della stessa categoria: "qualcosa che so". 

Inoltre **dimostrare di conoscere un segreto non significa necessariamente rivelarlo**. Esistono infatti diversi modi per fornire la prova, tra cui:
- **PAP** (Password Authentication Protocol): mostro la password all'identificatore
- **CHAP** (Challenge Handshake Authentication Protocol): mostro il risultato di un calcolo che dipende dalla password e da una challenge casuale, senza rivelare la password stessa (es. calcolo un hash della password concatenata con la challenge)
- **Zero Knowledge Proof**: dimostro una proprietà senza rivelare informazioni aggiuntive sul segreto, secondo la definizione e le assunzioni del protocollo.

CHAP non rappresenza una ZKP perché una coppia challenge-risposta può permettere all'attaccante di **verificare offline ipotesi sulla password**.  
Ad esempio osservando $R = F(p,C)$ può provare una password candidata $p'$ e controllare se $F(p', C) = R$ --> la password non viene trasmessa in chiaro, ma offre un modo di verificare tentativi.

### **PAP e CHAP**
Nel modello più semplice PAP, il client invia $(\text{UID}, p)$ al server. Il server allora individua l'account tramite UID, verifica la password e risponde con successo o fallimento.

PAP non protegge autonomamente la password durante il trasporto, se il canale è leggibile da un attaccante quindi anche la password lo è.  
Molti login seguono quindi una logica simile all'interno di un canale TLS (Transport Layer Security), come HTTPS, per cui la password è disponibile al server ma il traffico sulla rete cifrato. Ciò non significa comunque che poi il server debba memorizzare la password in chiaro: può memorizzarne un hash o altri derivati sicuri.

La struttura di un protocollo Challenge-Response **CHAP** è invece la seguente:
1. Il server genera la challenge $C$ e la invia al client
2. Il client calcola la risposta $R = F(p, C)$ e invia $\text{UID}, R$ al server
3. Il server recupera il segreto $p$ associato a $\text{UID}$, calcola $R' = F(p, C)$ e confronta $R'$ con $R$. Se coincidono, l'autenticazione ha successo.

Quindi il server non ricava la password dalla risposta: dispone già del materiale per verificarla. La challenge serve a rendere la risposta diversa per ogni sessione, così che un attaccante non possa riutilizzare una risposta intercettata.

Password $p$ e chiave segreta $K$ sono entrambi valori segreti, ma diversa entropia.  
Mentre una chiave uniforme di $n$ bit ha $2^n$ valori equiprobabili ($n$ bit di entropiaq); una password di $L$ caratteri scelti u.a.r. e indipendentemente in un alfabeto di $q$ simboli ha $q^L$ possibilità, ossia $L \log_2 q$ bit di entropia.

Tuttavia nella realtà le password scelte dalle persone raramente seguono quel modello: parole comuni, date e sostituzioni prevedibili come a -> 4 rendono alcuni tentativi molto più probabili di altri. I programmi di cracking provano prima proprio queste combinazioni.

**Meglio PAP o CHAP? Dipende dall'attaccante!**  
In generale non è sufficiente chiedere se un protocollo sia "più sicuro", ma **bisogna specificare quale attacco si sta considerando**. 

Ad esempio se si ha un attacco sul canale di trasmissione e il canale non è protetto, è chiaro che CHAP è migliore in quanto non espone direttamente la password (nonostante mantenga rischi come i tentativi offline contro password deboli).  
Se invece il canale è protetto ma l'attaccante ruba il database (e il server memorizza le password in chiaro), **diventano decisive le informazioni conservate dal server** per cui PAP risulta più sicuro.

#### PAP con password memorizzate tramite hash e salt
Il server può conservare un verificatore $v = H(p)$ invece che la password stessa. Quando riceve la password candidata $p'$, calcola $H(p')$ e verifica che sia uguale al risultato salvato $v$.

Il protocollo verso l'utente non cambia, ma il vantaggio è che chiaramente se il database viene bucato l'attaccante ruba $v$ ma non direttamente $p$, per cui dovrà provare password candidate e confrontare i risultati.  
Ciò attenua il danno ma non elimina gli attacchi, dato che password deboli possono essere indovinate rapidamente. Inoltre in implementazioni reali la generica $H$ va sostituita con una **funzione hash specifica per password**, introduciamo il concetto di **salt**.

Per ciascun account il server in realtà conserva:
$$(UID, s, v) \text{e} v = \text{PasswordHash}(p, s)$$
dove il salt $s$ è pubblico e viene generato in modo casuale, con dimensione sufficiente per evitare riutilizzi.

Quando un utente invia $p'$, il server recupera $s$ e verifica che $\text{PasswordHash}(p', s) = v$, il salt non deve essere ricordato dall'utente.  
Il vantaggio si vede confrontando due account con la stessa password:
$$v_1 = \text{PasswordHash}(p, s_1) \neq v_2 = \text{PasswordHash}(p, s_2)$$
In pratica con salt diversi i verificatori sono generalmente differenti, per cui l'attaccante non può riconoscere direttamente le password uguali né riutilizzare una tabella precalcolata di hash (rainbow table) per attaccare più account contemporaneamente (senza salt l'attaccante potrebbe calcolare tanti $H(p')$ con $p'$ password facili, e avendola già pronta gli basterebbe fare il confronto per trovare password corrette).

Il salt quindi:
- ostacola le precomputazioni, comprese le rainbow table
- impedisce di ammortizzare lo stesso calcolo su molti account con salt diversi

Tuttavia ovviamente non impedisce all'attaccante di provare password candidate per il singolo account, dato che l'attaccante conosce il salt.

#### Il problema database di CHAP
Per verificare la risposta del client $R = F(p, C)$, il server disporre di $p$, o comunque di un materiale equivalente sufficiente per eseguire il calcolo.

Conservare solo $H(p)$ in generale infatti non permette di ricostruire $F(p, C)$, in quanto una normale hash della password non offre operazioni invertibili o che producano automaticamente una risposta alla challenge.  
Il CHAP storico richiede infatti che il segreto sia disponibile al verificatore in forma utilizzabile. Nota che "disponibile" non implica necessariamente memorizzato senza cifratura su disco, ma il server deve comunque poterlo recuperare in chiaro per usarlo.

**Quindi mentre PAP in generale, su canale protetto, può far sì che il server usi un database contenente solo verificatori di password (come nell'esempio il loro hash), CHAP classico richiede direttamente un segreto utilizzabile per generare risposte.**

Una persona potrebbe ingenuamente pensare di usare invece che la password come segreto, l'hash della password: $v = H(p)$, e calcolare la risposta come $R = F(v, C)$. A questo punto il server potrebbe verificare usando solo $v$, tuttavia in questo modo **il valore $v$ è diventato una credenziale equivalente alla password** in quanto se un attaccante ruba $v$, può calcolare $F(v, C)$ per qualsiasi challenge $C$ e quindi autenticarsi come l'utente senza conoscere la password originale, che a quel punto non gli serve recuperare.

### Funzioni specifiche per password
Conosciamo un ottimo hash, SHA256. Perché non potremmo utilizzarlo per le password? Per via del fatto che **SHA256 è progettato per essere veloce, mentre noi vogliamo rendere costoso ogni tentativo di indovinare la password da parte dell'attaccante**. 

Se una password candidata richiede poco tempo per essere verificata, allora l'attaccante può provarne moltissime al secondo (es. vedi hardware specializzati per mining Bitcoin), e ricordiamo che nel caso CHAP l'attaccante può fare tentativi **offline** senza interagire con il server, quindi senza rischiare di essere bloccato.

Le funzioni specifiche per password aggiungono un costo regolabile per ogni tentativo. In particolare si presenta la funzione **bcrypt**, con salt e parametro **cost**. Il lavoro cresce approssimativamente come $2^{\text{cost}}$, quindi un cost=10 richiede circa 1000 volte più lavoro di cost=0.

In pratica quando un utente si registra il server genera un salt casuale e salva nel database il risultato della funzione bcrypt: $v = \text{bcrypt}(p, salt, \text{cost})$. Quando poi l'utente fa login inserendo la password $p'$, allora il server esegue bycript con lo stesso salt e cost.  
Il server paga quindi il costo di bycript una sola volta per login, mentre un eventuale attaccante che tenta di indovinare la password lo paga per ogni password candidata. 

Altri esempi di funzioni specifiche per password sono **PBKDF2** e **Argon2**, quest'ultima in particolare introduce un costo anche di memoria oltre a quello computazionale, per rendere costosi gli attacchi su hardware parallelo. 

In pratica quindi per conservare password serve un **salt affinché l'attaccante non possa riutilizzare calcoli precalcolati** e una **funzione specifica per password** che renda costoso ogni tentativo di indovinare la password per uno specifico account.

### Autenticazione reciproca e Reflection Attack
Finora abbiamo visto come far sì che una parte dimostri di conoscere un segreto. Nell'**autenticazione reciproca** vogliamo che **entrambe le parti lo dimostrino**.

Potremmo pensare ingenuamente di eseguire CHAP in entrambe le direzioni, con uno schema del tipo:
1. Il server invia una challenge $C_1$ al client
2. Il client risponde con $R_1 = F(p, C_1)$
3. Il client invia una challenge $C_2$ al server
4. Il server risponde con $R_2 = F(p, C_2)$

con stessa chiave segreta $K$ (sarebbe la password condivisa) usata in entrambe le direzioni.  
Il problema è che il protocollo permette all'attaccante di **ottenere da una parte la risposta che poi può presentare come propria prova dall'altra!**

Infatti il cosiddetto **Reflection Attack** funziona così. Supponiamo l'attaccante voglia autenticarsi presso un server senza conoscere $K$:
1. Il server gli invia $C_1$ e si aspetta $F(K, C_1)$
2. L'attaccante chiede al server di autenticarsi usando proprio $C_1$ come challenge
3. Il server risponde con $R_1 = F(K, C_1)$
4. L'attaccante copia quel valore e lo invia come risposta alla challenge originale, per cui il server accetterà l'autenticazione dell'attaccante.

Questo attacco ha senso dal momento che le due direzioni client-server tipicamente sono indipendenti, per cui non possiamo assumere un ordine di invio dei messaggi.

Ma una soluzione banale al problema sarebbe quella di **richiedere di usare challenge diverse $C_1 \neq C_2$**. Tuttavia questo non basta, vale solo all'interno di una singola sessione! 

Un Reflection Attack tra due sessioni infatti funziona come segue: l'attaccante si presenta al Client come fosse il server. Nella prima sessione invia $C_1$, e il client risponde con $C_2$ e $R_1 = F(K, C_1)$. L'attaccante dovrebbe completare la sessione con il client rispondendo con $R_2 = F(K, C_2)$, ma non conosce $K$, quindi banalmente **apre una seconda sessione con lo stesso client** e invia $C_2$ come challenge, per cui il client risponderà con $R_2 = F(K, C_2)$, che l'attaccante potrà copiare e inviare come risposta alla prima sessione ancora aperta.

Il punto è che quindi l'attaccante può intrecciare esecuzioni diverse! E l'idea per cui vi sia solo una sessione aperta alla volta non è ovviamente praticabile.

Anche se lo stato del protocollo ostacolasse la riflessione solo verso il Client, l'attaccante potrebbe aprire la seconda sessione verso il server (si parla in questo caso di **intertwining attack**): di nuovo dopo aver ottenuto $C_2$ dal client aprirebbe la sessione con il server e invierebbe $C_2$ come challenge, ottenendo $R_2 = F(K, C_2)$ che potrà usare per completare la prima sessione. Il difetto è esattamente lo stesso.

Un'idea potrebbe essere quella di **aggiungere una terza challenge $C_3$, ma non basta!**

<img src="imgs/attacl.png" width="300">

Qui l'idea è far dipendere ogni risposta da due valori (che rappresentano dei nonce, più che delle challenge). Però anche qui l'attaccante può creare una seconda sessione fingendosi server e bucare il protocollo:
1. Nella prima riceve $C_2$ e chiede $C_1$, ricevendo $R_1 = F(K, C_1, C_2)$ dal client
2. Apre la seconda inviando $C_2$, riceve $C_3$ dal client che risponde con $R_2 = F(K, C_2, C_3)$
3. L'attaccante quindi copia $R_2$ e lo invia come risposta alla prima sessione, per cui il client lo accetterà come valido.

Quindi aggiungere nonce non rende automaticamente sicuro il protocollo, conta come vengono legati crittograficamente!

#### La soluzione
La proposta risolutiva usa solo $C_1$ e $C_2$, ma in ordine inverso:

<img src="imgs/attacl2.png" width="300">

Entrambe le direzioni **dipendono dalla stessa coppia di challenge**, ma l'ordine distingue le due direzioni.  

Per capire perché blocca l'intertwining attack, supponiamo che l'attaccante cerchi $F(K, C_1, C_2)$ (quindi aveva inviato $C_1$ al client). Allora nella seconda sessione può inviare $C_2$, ma il client sceglie una nuova challenge $C_3$ e risponde con $R_2 = F(K, C_2, C_3)$, che non risponde alla prima sessione che richiedeva $C_1$ nella seconda posizione, per cui l'attaccante non può completarla.

Questa proprietà è detta **cryptographic binding**: entrambe le direzioni (server e client) contengono dentro il calcolo crittografico ($R_{\text{client} = F(K, C_1, C_2)}$ e $R_{\text{server} = F(K, C_2, C_1)}$) **entrambe le challenge della sessione**, con l'ordine diverso che distingue le sessioni.  
Se l'attaccante apre una seconda sessione usando $C_2$ come prima challenge, riceve $C_3$ come challenge che distingue la nuova sessione, per cui non può completare l'attacco.

Quindi la sicurezza sta nel fatto che le direzioni sono legate crittograficamente alla stessa sessione, non soltanto nell'ordine di esecuzione del codice. 

Chiaramente quanto visto è una semplificazione, in una specifica concreta bisognerebbe definire anche codifica dei campi, identità, ruoli, freschezza e verifiche. 